In [ ]:
from pathlib import Path

import ee
import geemap
import geopandas as gpd
import xarray as xr
import xee

try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()


In [ ]:
states_shp = Path("shp/SAsiaFinalD.shp")
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

if not states_shp.exists():
    raise FileNotFoundError(f"Missing shapefile: {states_shp.resolve()}")

gdf = gpd.read_file(states_shp, encoding="latin1").to_crs("EPSG:4326")
roi = geemap.geopandas_to_ee(gdf)
roi_geom = roi.geometry()

print(f"AOI features: {len(gdf)}")
print(f"AOI bounds: {gdf.total_bounds}")


In [ ]:
aoi_map = geemap.Map(basemap="SATELLITE")
aoi_map.centerObject(roi_geom, 4)
aoi_map.addLayer(roi.style(color="red", fillColor="00000000", width=2), {}, "AOI")
aoi_map


In [ ]:
dataset_id = "NASA/GIMMS/3GV0"
start_year = 1981
end_year = 2013
export_scale_m = 10000  # 10 km
band_name = "ndvi"

gimms = ee.ImageCollection(dataset_id).select(band_name).filterBounds(roi_geom)
native_projection = ee.Image(gimms.first()).select(band_name).projection()
year_list = ee.List.sequence(start_year, end_year)

def annual_ndvi(year):
    year = ee.Number(year).toInt()
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, "year")

    image = (
        gimms.filterDate(start, end)
        .mean()
        .rename(band_name)
        .clip(roi_geom)
        .set("year", year)
        .set("system:time_start", start.millis())
        .set("system:time_end", end.advance(-1, "day").millis())
    )
    return image

annual_collection = ee.ImageCollection.fromImages(year_list.map(annual_ndvi))
annual_collection


In [ ]:
preview = annual_collection.first()
vis = {"min": 0.0, "max": 0.8, "palette": ["brown", "yellow", "green"]}

preview_map = geemap.Map(basemap="SATELLITE")
preview_map.centerObject(roi_geom, 4)
preview_map.addLayer(preview, vis, f"Annual mean NDVI {start_year}")
preview_map.addLayer(roi.style(color="red", fillColor="00000000", width=2), {}, "AOI")
preview_map


In [ ]:
ds = xr.open_dataset(
    annual_collection,
    engine="ee",
    geometry=roi_geom,
    projection=native_projection,
    scale=export_scale_m,
)

ds = ds.sortby("time")
ds[band_name].attrs.update({
    "long_name": "Annual mean NDVI",
    "source": dataset_id,
    "units": "1",
    "resolution_m": export_scale_m,
})
ds.attrs.update({
    "title": "GIMMS NDVI annual mean clipped to AOI",
    "dataset": dataset_id,
    "time_coverage_start": f"{start_year}-01-01",
    "time_coverage_end": f"{end_year}-12-31",
})

ds


In [ ]:
output_nc = output_dir / "gimms_ndvi_annual_aoi_10km_1981_2013.nc"
encoding = {band_name: {"zlib": True, "complevel": 4}}

ds.to_netcdf(output_nc, encoding=encoding)
print(f"Saved: {output_nc.resolve()}")


In [ ]:
viirs_dataset_id = "NOAA/CDR/VIIRS/NDVI/V1"
viirs_start_year = 2014
viirs_end_year = 2025  # last full year; 2026 is partial in Earth Engine availability
viirs_export_scale_m = 10000  # 10 km
viirs_band_name = "NDVI"
viirs_scale_factor = 0.0001

viirs = ee.ImageCollection(viirs_dataset_id).select(viirs_band_name).filterBounds(roi_geom)
viirs_projection = ee.Image(viirs.first()).select(viirs_band_name).projection()
viirs_year_list = ee.List.sequence(viirs_start_year, viirs_end_year)

def annual_viirs_ndvi(year):
    year = ee.Number(year).toInt()
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, "year")

    image = (
        viirs.filterDate(start, end)
        .mean()
        .multiply(viirs_scale_factor)
        .rename("ndvi")
        .clip(roi_geom)
        .set("year", year)
        .set("system:time_start", start.millis())
        .set("system:time_end", end.advance(-1, "day").millis())
    )
    return image

annual_collection_viirs = ee.ImageCollection.fromImages(viirs_year_list.map(annual_viirs_ndvi))
annual_collection_viirs


In [ ]:
viirs_preview = annual_collection_viirs.first()
viirs_vis = {"min": 0.0, "max": 0.8, "palette": ["brown", "yellow", "green"]}

viirs_map = geemap.Map(basemap="SATELLITE")
viirs_map.centerObject(roi_geom, 4)
viirs_map.addLayer(viirs_preview, viirs_vis, f"Annual mean VIIRS NDVI {viirs_start_year}")
viirs_map.addLayer(roi.style(color="red", fillColor="00000000", width=2), {}, "AOI")
viirs_map


In [ ]:
ds_viirs = xr.open_dataset(
    annual_collection_viirs,
    engine="ee",
    geometry=roi_geom,
    projection=viirs_projection,
    scale=viirs_export_scale_m,
)

ds_viirs = ds_viirs.sortby("time")
ds_viirs["ndvi"].attrs.update({
    "long_name": "Annual mean NDVI",
    "source": viirs_dataset_id,
    "units": "1",
    "resolution_m": viirs_export_scale_m,
})
ds_viirs.attrs.update({
    "title": "VIIRS NDVI annual mean clipped to AOI",
    "dataset": viirs_dataset_id,
    "time_coverage_start": f"{viirs_start_year}-01-01",
    "time_coverage_end": f"{viirs_end_year}-12-31",
})

ds_viirs


In [ ]:
output_viirs_nc = output_dir / "viirs_ndvi_annual_aoi_10km_2014_2025.nc"
encoding_viirs = {"ndvi": {"zlib": True, "complevel": 4}}

ds_viirs.to_netcdf(output_viirs_nc, encoding=encoding_viirs)
print(f"Saved: {output_viirs_nc.resolve()}")


In [ ]:
avhrr_dataset_id = "NOAA/CDR/AVHRR/NDVI/V5"
avhrr_start_year = 1981
avhrr_end_year = 2013
avhrr_export_scale_m = 10000  # 10 km
avhrr_band_name = "NDVI"
avhrr_scale_factor = 0.0001

avhrr = ee.ImageCollection(avhrr_dataset_id).select(avhrr_band_name).filterBounds(roi_geom)
avhrr_projection = ee.Image(avhrr.first()).select(avhrr_band_name).projection()
avhrr_year_list = ee.List.sequence(avhrr_start_year, avhrr_end_year)

def annual_avhrr_ndvi(year):
    year = ee.Number(year).toInt()
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, "year")

    image = (
        avhrr.filterDate(start, end)
        .mean()
        .multiply(avhrr_scale_factor)
        .rename("ndvi")
        .clip(roi_geom)
        .set("year", year)
        .set("system:time_start", start.millis())
        .set("system:time_end", end.advance(-1, "day").millis())
    )
    return image

annual_collection_avhrr = ee.ImageCollection.fromImages(avhrr_year_list.map(annual_avhrr_ndvi))
annual_collection_avhrr


In [ ]:
avhrr_preview = annual_collection_avhrr.first()
avhrr_vis = {"min": 0.0, "max": 0.8, "palette": ["brown", "yellow", "green"]}

avhrr_map = geemap.Map(basemap="SATELLITE")
avhrr_map.centerObject(roi_geom, 4)
avhrr_map.addLayer(avhrr_preview, avhrr_vis, f"Annual mean AVHRR NDVI {avhrr_start_year}")
avhrr_map.addLayer(roi.style(color="red", fillColor="00000000", width=2), {}, "AOI")
avhrr_map


In [ ]:
ds_avhrr = xr.open_dataset(
    annual_collection_avhrr,
    engine="ee",
    geometry=roi_geom,
    projection=avhrr_projection,
    scale=avhrr_export_scale_m,
)

ds_avhrr = ds_avhrr.sortby("time")
ds_avhrr["ndvi"].attrs.update({
    "long_name": "Annual mean NDVI",
    "source": avhrr_dataset_id,
    "units": "1",
    "resolution_m": avhrr_export_scale_m,
})
ds_avhrr.attrs.update({
    "title": "AVHRR NOAA CDR NDVI annual mean clipped to AOI",
    "dataset": avhrr_dataset_id,
    "time_coverage_start": f"{avhrr_start_year}-01-01",
    "time_coverage_end": f"{avhrr_end_year}-12-31",
})

ds_avhrr


In [ ]:
output_avhrr_nc = output_dir / "avhrr_ndvi_annual_aoi_10km_1981_2013.nc"
encoding_avhrr = {"ndvi": {"zlib": True, "complevel": 4}}

ds_avhrr.to_netcdf(output_avhrr_nc, encoding=encoding_avhrr)
print(f"Saved: {output_avhrr_nc.resolve()}")


In [ ]:
target_dims = [dim for dim in ["lat", "lon"] if dim in ds_avhrr.dims and dim in ds_viirs.dims]

if target_dims:
    interp_coords = {dim: ds_avhrr.coords[dim] for dim in target_dims}
    ds_viirs_aligned = ds_viirs.interp(interp_coords, method="nearest")
else:
    ds_viirs_aligned = ds_viirs.copy()

combined_ndvi_ds = xr.concat(
    [ds_avhrr, ds_viirs_aligned],
    dim="time",
    data_vars="minimal",
    coords="minimal",
    compat="override",
).sortby("time")

combined_ndvi_ds["ndvi"].attrs.update({
    "long_name": "Annual mean NDVI",
    "source": "NOAA/CDR/AVHRR/NDVI/V5 + NOAA/CDR/VIIRS/NDVI/V1",
    "units": "1",
    "resolution_m": 10000,
})
combined_ndvi_ds.attrs.update({
    "title": "Combined NOAA CDR annual NDVI clipped to AOI",
    "dataset": "NOAA/CDR/AVHRR/NDVI/V5 + NOAA/CDR/VIIRS/NDVI/V1",
    "time_coverage_start": "1981-01-01",
    "time_coverage_end": "2025-12-31",
})

combined_ndvi_ds


In [ ]:
output_combined_nc = output_dir / "noaa_cdr_ndvi_annual_aoi_10km_1981_2025.nc"
encoding_combined = {"ndvi": {"zlib": True, "complevel": 4}}

combined_ndvi_ds.to_netcdf(output_combined_nc, encoding=encoding_combined)
print(f"Saved: {output_combined_nc.resolve()}")



Better long-term fix in Python

The cleanest solution is to fix it before writing the NetCDF. In xarray, reorder the dimensions before export:

da = da.transpose("time", "lat", "lon")

da.to_netcdf("noaa_cdr_ndvi_annual_aoi_10km_1981_2025.nc")
